# Dashboard feature-set export v1

目标：验证 `src/dashboard` 能读取模型矩阵公开 sidecar，生成 dashboard 使用的统一 JSON payload。

边界：dashboard 不直接读取 raw predictions 或 daily backtest，只展示汇总、regime、qlib risk、模型差异诊断和 artifact 指针。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import write_model_matrix_dashboard_export

CARD = REPO_ROOT / 'research/dashboard_feature_set_export_v1'
SOURCE_CARD = REPO_ROOT / 'research/factor_feature_set_matrix_v1'
CARD.mkdir(parents=True, exist_ok=True)
payload = write_model_matrix_dashboard_export(SOURCE_CARD, output_path=CARD / 'dashboard_export.json', top_n=5)
payload.keys()

dict_keys(['schema_version', 'artifact_id', 'artifact_type', 'summary', 'top_models', 'feature_set_summary', 'regime_metrics', 'window_metrics', 'qlib_risk_metrics', 'model_effect_deltas', 'artifacts'])

In [2]:
payload['top_models']

[{'rank': 1,
  'model': 'random_forest_clf__all_features',
  'base_model': 'random_forest_clf',
  'feature_set': 'all_features',
  'return_pct': 49.032787769965715,
  'benchmark_return_pct': 62.1472468532841,
  'relative_return_pct': -13.114459083318389,
  'max_drawdown_pct': 21.677243377022705,
  'sharpe': 3.8534654423822032,
  'avg_position': 0.7407635422717518,
  'turnover': 3.1283151044281827,
  'stability_score': 54.214257763652995},
 {'rank': 2,
  'model': 'hist_gradient_mse__top_ic_factors',
  'base_model': 'hist_gradient_mse',
  'feature_set': 'top_ic_factors',
  'return_pct': 47.73460950926427,
  'benchmark_return_pct': 62.1472468532841,
  'relative_return_pct': -14.412637344019837,
  'max_drawdown_pct': 24.12777644362798,
  'sharpe': 3.798898276020224,
  'avg_position': 0.7140206199854767,
  'turnover': 2.6146788794514384,
  'stability_score': 50.33451395602866},
 {'rank': 3,
  'model': 'ridge_mse__all_features',
  'base_model': 'ridge_mse',
  'feature_set': 'all_features',
 

In [3]:
payload['feature_set_summary']

[{'feature_set': 'all_features',
  'variant_count': 3,
  'best_model': 'random_forest_clf__all_features',
  'best_base_model': 'random_forest_clf',
  'avg_return_pct': 45.53785633690822,
  'avg_relative_return_pct': -16.609390516375882,
  'avg_stability_score': 47.63418055404903},
 {'feature_set': 'top_ic_factors',
  'variant_count': 3,
  'best_model': 'hist_gradient_mse__top_ic_factors',
  'best_base_model': 'hist_gradient_mse',
  'avg_return_pct': 46.5956591007232,
  'avg_relative_return_pct': -15.551587752560897,
  'avg_stability_score': 46.812444861592724},
 {'feature_set': 'ic_weighted_composite',
  'variant_count': 3,
  'best_model': 'random_forest_clf__ic_weighted_composite',
  'best_base_model': 'random_forest_clf',
  'avg_return_pct': 43.05944249510467,
  'avg_relative_return_pct': -19.087804358179422,
  'avg_stability_score': 39.31050534112193}]

In [4]:
{
    'artifacts': payload['artifacts'],
    'raw_predictions_exposed': 'predictions' in payload['artifacts'],
    'daily_backtest_exposed': 'daily_backtest' in payload['artifacts'],
    'qlib_risk_rows': len(payload['qlib_risk_metrics']),
    'model_effect_delta_rows': len(payload['model_effect_deltas']),
}

{'artifacts': {'summary': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/analysis_summary.json',
  'model_comparison': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/model_comparison.csv',
  'regime_metrics': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/regime_metrics.csv',
  'window_metrics': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/window_metrics.csv',
  'qlib_risk_metrics': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/qlib_risk_metrics.csv',
  'model_effect_deltas': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/model_effect_deltas.csv',
  'feature_set_specs': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/feature_set_specs.json',
  'factor_ic': '/Users/rziiiii/dev/workspace/research/factor_feature_set_matrix_v1/factor_ic.csv'},
 'raw_predictions_exposed': False,
 'daily_backtest_exposed': False,
 'qlib_risk_rows': 108,
 'model_

In [5]:
summary = {
    'schema_version': 'dashboard_export_review.v1',
    'source_card': str(SOURCE_CARD.relative_to(REPO_ROOT)),
    'dashboard_export': 'dashboard_export.json',
    'top_model': payload['top_models'][0]['model'],
    'feature_set_summary_count': len(payload['feature_set_summary']),
    'qlib_risk_rows': len(payload['qlib_risk_metrics']),
    'model_effect_delta_rows': len(payload['model_effect_deltas']),
    'raw_predictions_exposed': 'predictions' in payload['artifacts'],
    'daily_backtest_exposed': 'daily_backtest' in payload['artifacts'],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'dashboard_export_review.v1',
 'source_card': 'research/factor_feature_set_matrix_v1',
 'dashboard_export': 'dashboard_export.json',
 'top_model': 'random_forest_clf__all_features',
 'feature_set_summary_count': 3,
 'qlib_risk_rows': 108,
 'model_effect_delta_rows': 54,
 'raw_predictions_exposed': False,
 'daily_backtest_exposed': False}

## 中文结论

- 做了什么：把 `factor_feature_set_matrix_v1` 的公开 sidecar 转换为 dashboard JSON export。
- 核心指标：Top model 为 `random_forest_clf__all_features`；feature-set 聚合数量 3；qlib risk 行数 108；模型差异诊断行数 54。
- 展示范围：dashboard payload 包含 summary、top_models、feature_set_summary、regime/window metrics、qlib_risk_metrics、model_effect_deltas 和 artifact 指针。
- 数据边界：不暴露 `predictions.csv` 或 `daily_backtest.csv`，符合 dashboard 只读公开 export 的边界。
- 是否晋升：否。本轮是 dashboard export 框架验证，不代表模型/因子方案进入 production。
- 下一步：补 schedule job，把模型矩阵运行后自动生成 dashboard export，并接通知/锁/重试。